# Alignment Module — Run on RunPod / Jupyter

This notebook runs the full alignment experiment: load task (SummEval), split data, populate cache (run TrustScore once per sample), run optimizers A/B/C, and evaluate best configs on train/val/test. All results are saved under `results_dir/run_<id>/`.

**Setup:** Set `results_dir` and `cache_dir` to a persistent path on RunPod (e.g. `/workspace/alignment/results`).

In [ ]:
import sys
import os

# Add project root so we can import alignment and pipeline
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)
print("Project root:", project_root)

In [ ]:
# Configuration — change these for your run (e.g. on RunPod use persistent paths)
RESULTS_DIR = "alignment/results"
CACHE_DIR = "alignment/cache"
TASK_NAME = "summeval"
METHOD = "all"  # "a", "b", "c", or "all"
MAX_SAMPLES = 50
MAX_EVALS = 30
TRAIN_RATIO = 0.6
VAL_RATIO = 0.2
RANDOM_SEED = 42
REFRESH_CACHE = False
SPLITS_MANIFEST_PATH = None  # Set to path/to/splits.json to reuse splits from a prior run
API_KEY = None  # Set if using OpenAI (not needed when USE_LLAMA=True)
# LLaMA: use VLLM with HuggingFace model ID, or local LLaMA with MODEL_PATH
USE_LLAMA = True
LLAMA_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_PATH = None  # Set to local path for LLaMA provider; None uses VLLM with LLAMA_MODEL
NUM_JUDGES_PER_CATEGORY = 3
MAX_TOKENS = 4096  # Max tokens for LLM generation (increase if spans get truncated)
TEMPERATURE = 0.1

In [ ]:
from alignment.run_alignment import run

result = run(
    task_name=TASK_NAME,
    method=METHOD,
    max_samples=MAX_SAMPLES,
    max_evals=MAX_EVALS,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    random_seed=RANDOM_SEED,
    cache_dir=CACHE_DIR,
    results_dir=RESULTS_DIR,
    refresh_cache=REFRESH_CACHE,
    splits_manifest_path=SPLITS_MANIFEST_PATH,
    api_key=API_KEY,
    use_llama=USE_LLAMA,
    llama_model=LLAMA_MODEL,
    model_path=MODEL_PATH,
    num_judges_per_category=NUM_JUDGES_PER_CATEGORY,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
)

print("Run ID:", result["run_id"])
print("Run dir:", result["run_dir"])
print("Report:", result["alignment_report_path"])

## Inspect splits

In [ ]:
import json

splits_path = os.path.join(result["run_dir"], "splits.json")
if os.path.exists(splits_path):
    with open(splits_path) as f:
        splits = json.load(f)
    print("Train:", splits.get("n_train"), "Val:", splits.get("n_val"), "Test:", splits.get("n_test"))
    print("Train IDs (first 5):", splits.get("train_ids", [])[:5])
else:
    print("Splits file not found (run may have used existing splits manifest).")

## Inspect results

In [ ]:
with open(result["alignment_results_path"]) as f:
    alignment_results = json.load(f)

with open(os.path.join(result["run_dir"], "best_configs.json")) as f:
    best_configs = json.load(f)

import pandas as pd

# Correlation table (Pearson / Spearman / Kendall)
rows = []
for method, res in alignment_results.items():
    rows.append({
        "method": method,
        "train_pearson": res.get("train_pearson"),
        "train_spearman": res.get("train_spearman"),
        "train_kendall": res.get("train_kendall"),
        "val_pearson": res.get("val_pearson"),
        "val_spearman": res.get("val_spearman"),
        "val_kendall": res.get("val_kendall"),
        "test_pearson": res.get("test_pearson"),
        "test_spearman": res.get("test_spearman"),
        "test_kendall": res.get("test_kendall"),
    })
print("=== Correlation Results ===")
display(pd.DataFrame(rows))

# Category score statistics (mean / median / std for T, E, B, aggregated)
stat_rows = []
for method, res in alignment_results.items():
    cs = res.get("category_stats", {})
    row = {"method": method}
    for cat in ["T", "E", "B", "aggregated"]:
        s = cs.get(cat, {})
        row[f"{cat}_mean"] = s.get("mean")
        row[f"{cat}_median"] = s.get("median")
        row[f"{cat}_std"] = s.get("std")
    stat_rows.append(row)
print("\n=== Category Score Statistics ===")
display(pd.DataFrame(stat_rows))

In [ ]:
print("Best configs (summary):")
for method, data in best_configs.items():
    print(method, ":", data.get("config_dict", {}))

## Per-sample scores

In [ ]:
sample_scores_path = result.get("sample_scores_path")
if sample_scores_path and os.path.exists(sample_scores_path):
    with open(sample_scores_path) as f:
        sample_scores = json.load(f)
    # Show per-sample scores for the default (untuned) config
    if "default" in sample_scores:
        df_default = pd.DataFrame(sample_scores["default"])
        print(f"Per-sample scores (default config): {len(df_default)} samples")
        display(df_default.describe())
        display(df_default.head(10))
else:
    print("Per-sample scores file not found.")